In [1]:
#import pyspark
import findspark

In [2]:
findspark.find()

'D:\\Software\\spark-4.0.0-bin-hadoop3\\spark-4.0.0-bin-hadoop3'

In [3]:
#initiate spark
import pyspark
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
c = pyspark.SparkConf().setAppName("test_app").setMaster("local")
sc = pyspark.SparkContext(conf = c)
spark = SparkSession(sc)

In [9]:
from pyspark.sql.functions import *

In [40]:
data = spark.read.csv("D:\Data Engineering\Data Set\Sample - Superstore v1.csv",inferSchema=None, header = True, escape ='"')

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\gunja\AppData\Local\Temp\ipykernel_28300\1543472995.py:1: SyntaxWarning: invalid escape sequence '\D'
  data = spark.read.csv("D:\Data Engineering\Data Set\Sample - Superstore v1.csv",inferSchema=None, header = True, escape ='"')


In [41]:
data.show(5)

+------+--------------+----------+----------+--------------+-----------+-------------+-----------+--------------+-------------+----------+-----------+-------+---------------+----------+------------+--------------------+---------+--------+--------+----------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|Customer Name|    Segment|Country/Region|         City|     State|Postal Code| Region|     Product ID|  Category|Sub-Category|        Product Name|    Sales|Quantity|Discount|    Profit|
+------+--------------+----------+----------+--------------+-----------+-------------+-----------+--------------+-------------+----------+-----------+-------+---------------+----------+------------+--------------------+---------+--------+--------+----------+
|  2698|CA-2016-145317|18-03-2016|23-03-2016|Standard Class|   SM-20320|  Sean Miller|Home Office| United States| Jacksonville|   Florida|      32216|  South|TEC-MA-10002412|Technology|    Machines|Cisco TelePresenc...| 226

In [42]:
data.printSchema()

root
 |-- Row ID: string (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: string (nullable = true)



In [26]:
#data = data.withColumn("Sales", col("Sales").cast("double"))
#data = data.withColumn("Quantity", col("Quantity").cast("int"))
#data = data.withColumn("Discount", col("Discount").cast("double"))
#data = data.withColumn("Profit", col("Profit").cast("double"))

In [46]:
data1 = data.select("Row ID","Segment","Region","Category","Sales","Profit","Quantity") # selected few Column

In [47]:
data1.filter(col("Row ID")==6076).show(5)


+------+---------+------+---------------+------+------+--------+
|Row ID|  Segment|Region|       Category| Sales|Profit|Quantity|
+------+---------+------+---------------+------+------+--------+
|  6076|Corporate|  West|Office Supplies|270.62|2.7062|       2|
+------+---------+------+---------------+------+------+--------+



In [48]:
data1.show(5)

+------+-----------+-------+----------+---------+----------+--------+
|Row ID|    Segment| Region|  Category|    Sales|    Profit|Quantity|
+------+-----------+-------+----------+---------+----------+--------+
|  2698|Home Office|  South|Technology| 22638.48|-1811.0784|       6|
|  6827|  Corporate|Central|Technology| 17499.95|  8399.976|       5|
|  8154|   Consumer|   West|Technology| 13999.96| 6719.9808|       4|
|  2624|Home Office|   East|Technology|11199.968| 3919.9888|       4|
|  4191|   Consumer|   East|Technology| 10499.97| 5039.9856|       3|
+------+-----------+-------+----------+---------+----------+--------+
only showing top 5 rows


In [49]:
# 1col 1 aggr
data1.groupby("Segment").agg(sum("Sales")).show(5)

+-----------+-----------------+
|    Segment|       sum(Sales)|
+-----------+-----------------+
|   Consumer|1161401.344999977|
|Home Office|429653.1485000006|
|  Corporate|706146.3667999947|
+-----------+-----------------+



In [50]:
# 2col 1 aggr
data1.groupby("Segment","Region").agg(sum("Sales")).show(5)

+-----------+-------+------------------+
|    Segment| Region|        sum(Sales)|
+-----------+-------+------------------+
|Home Office|   West|136721.77699999965|
|  Corporate|   West|225855.27449999994|
|   Consumer|   West|362880.77300000144|
|   Consumer|   East|350908.16700000095|
|Home Office|Central| 91212.64399999991|
+-----------+-------+------------------+
only showing top 5 rows


In [60]:
# 1 col 2 agg
data1.groupby("Region","Segment").agg(sum("Sales").alias("TotalSales"),sum("Profit").alias("TotalProfit")).show(5)

+-------+-----------+------------------+------------------+
| Region|    Segment|        TotalSales|       TotalProfit|
+-------+-----------+------------------+------------------+
|Central|Home Office| 91212.64399999991|12438.412399999997|
|   West|Home Office|136721.77699999965|16530.414999999957|
|  South|Home Office|        74255.0015|4620.6343000000015|
|   East|Home Office| 127463.7259999999| 26709.21680000001|
|  South|   Consumer|195580.97100000046|26913.572799999987|
+-------+-----------+------------------+------------------+
only showing top 5 rows


In [63]:
# 2col 1 aggr
data1.groupby("Segment","Region").agg(sum("Sales").alias("Total_sales")).sort(desc("Total_sales")).show(5)

+---------+-------+------------------+
|  Segment| Region|       Total_sales|
+---------+-------+------------------+
| Consumer|   West|362880.77300000144|
| Consumer|   East|350908.16700000095|
| Consumer|Central|252031.43400000004|
|Corporate|   West|225855.27449999994|
|Corporate|   East|200409.34699999998|
+---------+-------+------------------+
only showing top 5 rows
